# 02 Pipeline

FabricOps uses one canonical pipeline shape: **0. Environment → E. Extract → T. Transform → L. Load**.

This template mixes validated Live components with newer Preview components in their correct lifecycle position instead of maintaining separate notebook versions. Live code cells are expanded and runnable. Preview cells are collapsed and wrapped in a triple-quoted block so `Run all` keeps the validated baseline path safe until those components are enabled deliberately.

Lakehouse is the preferred Spark-heavy path because Delta data is directly available through OneLake. FabricOps also supports Fabric Warehouse sources and targets where the pipeline requires them.

## Component maturity

- **Live** — validated baseline component; expanded by default.
- **Preview** — implemented governed capability; collapsed and disabled by default until deliberately enabled.

To enable a Preview example, expand the cell and remove the opening and closing triple quotes after confirming the required Governance metadata exists.

## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table records the releases manually tested with the baseline notebook flow in Microsoft Fabric.

| FabricOps release | Tested by | Date tested |
|---|---|---|
| v0.2.0 | Voyce | 6 Aug 2026 |

# 0. Environment

### Live — Resolve the active FabricOps environment
Run `00_env_config` before continuing. It establishes Development or Production, configured Fabric stores, schemas, runtime context, widgets, and metadata routing.

In [ ]:
%run 00_env_config

### Live — Import baseline and Preview functions
Imports stay together so the same notebook can move from the Live baseline into the Preview governed lifecycle without changing its public function surface.

In [ ]:
from fabricops_kit import (
    # Live IO — FabricOps v0.1.0 onwards
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_query,
    read_warehouse_table,
    write_lakehouse_table,
    write_warehouse_table,
    # Live profiling — FabricOps v0.2.0 onwards
    profile_dataframe,
    profile_frequency_distribution,
    profile_and_register_table,
    # Preview governed runtime
    read_pipeline_prep,
    write_pipeline_prep,
    observe_table,
    check_schema,
    check_freshness,
    check_changes,
    check_dq,
    widget_select_data_contract,
    widget_view_catalogue,
)

# E. Extract

Choose the Live source example that matches the workload. The Preview governed source block follows the same physical readers but adds observation, Guardrails, and `skip` / `full` / `incremental` scope preparation before the business-data read.

### Live — Read Excel from a Lakehouse

In [ ]:
# Run this example only when Excel is the selected source.
# source_df = read_lakehouse_excel(
#     "excel_file_demo.xlsx", target="source", sheet_name="products", spark_session=spark,
# )
# display(source_df)
# display(profile_dataframe(source_df))
# display(profile_frequency_distribution(source_df))

### Live — Read Parquet from a Lakehouse

In [ ]:
# Run this example only when Parquet is the selected source.
# source_df = read_lakehouse_parquet(
#     "parquet_file_demo.parquet", target="source", spark_session=spark,
# )
# display(source_df)

### Live — Read CSV from a Lakehouse

In [ ]:
SOURCE_TARGET = "source"
source_df = read_lakehouse_csv(
    "lakehouse_data_demo.csv",
    target=SOURCE_TARGET,
    spark_session=spark,
    header=True,
    inferSchema=True,
)
display(source_df)

### Live — Read a Warehouse table or SQL query
Use `read_warehouse_table()` when the DataFrame represents the complete physical table. Use `read_warehouse_query()` for caller-defined SQL, but do not register a filtered, joined, or aggregated query result as the complete profile of one physical source table.

In [ ]:
# Complete-table Warehouse example. Run when Warehouse is the selected source.
# SOURCE_TARGET = "product"
# SOURCE_SCHEMA = "demo"
# SOURCE_TABLE_NAME = "email_logs_dummy"
# source_df = read_warehouse_table(
#     target=SOURCE_TARGET, schema=SOURCE_SCHEMA, table_name=SOURCE_TABLE_NAME, spark_session=spark,
# )
# source_profile_df = profile_and_register_table(
#     source_df, profile_role="source", target=SOURCE_TARGET,
#     schema=SOURCE_SCHEMA, table_name=SOURCE_TABLE_NAME,
# )

### Preview — Governed source preparation, Guardrails, and incremental load
This block is in the canonical Extract position. It resolves the selected/current contract context, observes source changes, prepares `skip` / `full` / `incremental` scope, evaluates source Guardrails, performs the explicit physical read, and protects full-table profiling from incremental slices.

In [ ]:
"""
PREVIEW — Enable only after the source and target are registered and the required Governance metadata exists.

SOURCE_TARGET = "source"
SOURCE_SCHEMA = "dbo"
SOURCE_TABLE_NAME = "student_enrolment"
TARGET_TARGET = "unified"
TARGET_SCHEMA = "demo"
TARGET_TABLE_NAME = "Laptop_Inventory"
TARGET_LOAD_STRATEGY = "scd1"
TARGET_LOAD_PARAMETERS = {"key_columns": ["student_id"]}

source_validation = widget_select_data_contract(
    SOURCE_TABLE_NAME, target=SOURCE_TARGET, schema=SOURCE_SCHEMA,
)
target_validation = widget_select_data_contract(
    TARGET_TABLE_NAME, target=TARGET_TARGET, schema=TARGET_SCHEMA,
)

read_prep = read_pipeline_prep(
    source_table_name=SOURCE_TABLE_NAME,
    source_target=SOURCE_TARGET,
    source_schema=SOURCE_SCHEMA,
    target_table_name=TARGET_TABLE_NAME,
    target=TARGET_TARGET,
    schema=TARGET_SCHEMA,
    load_strategy=TARGET_LOAD_STRATEGY,
    load_strategy_parameters=TARGET_LOAD_PARAMETERS,
)

schema_result = check_schema(target=SOURCE_TARGET, schema=SOURCE_SCHEMA, table_name=SOURCE_TABLE_NAME)
freshness_result = check_freshness(read_prep["observation"])
if not all(result["can_continue"] for result in (schema_result, freshness_result, read_prep["changes"])):
    raise RuntimeError("A pre-read source Guardrail blocked the pipeline.")

if read_prep["read_strategy"] == "skip":
    source_df = None
    source_profile_df = None
    print("No source change; source read, transformation and target write skipped.")
else:
    source_df = read_lakehouse_table(
        target=SOURCE_TARGET, schema=SOURCE_SCHEMA, table_name=SOURCE_TABLE_NAME, spark_session=spark,
    )
    if read_prep["read_strategy"] == "incremental":
        from pyspark.sql import functions as F
        source_df = source_df.where(
            F.col(read_prep["partition_column"]).isin(read_prep["partition_values"])
        )

    dq_result = check_dq(source_df, SOURCE_TABLE_NAME, target=SOURCE_TARGET, schema=SOURCE_SCHEMA)
    display(dq_result["summary"])
    if not dq_result["can_continue"]:
        raise RuntimeError("A DQ Guardrail blocked the pipeline.")

    if read_prep["read_strategy"] == "full":
        source_profile_df = profile_and_register_table(
            source_df, profile_role="source", target=SOURCE_TARGET,
            schema=SOURCE_SCHEMA, table_name=SOURCE_TABLE_NAME,
        )
    else:
        source_profile_df = None  # Keep the latest complete source profile.
"""

# T. Transform

### Live — User-defined transformation
FabricOps governs the boundaries around ETL rather than replacing business logic. Keep joins, filters, derivations, aggregations, enrichment, and reshaping visible here.

In [ ]:
from pyspark.sql.functions import current_timestamp
transformed_df = source_df.withColumn("ingested_at_utc", current_timestamp())
display(transformed_df)

# L. Load

Use the Live Lakehouse or Warehouse writer example for the validated baseline. The Preview governed-load block shows how target Guardrails and `write_pipeline_prep()` wrap the same explicit physical write.

### Live — Write, read back, and profile a Lakehouse target

In [ ]:
TARGET_TARGET = "unified"
TARGET_SCHEMA = "demo"
TARGET_TABLE_NAME = "Laptop_Inventory"

write_lakehouse_table(
    df=transformed_df,
    target=TARGET_TARGET,
    schema=TARGET_SCHEMA,
    table_name=TARGET_TABLE_NAME,
    mode="overwrite",
)
target_df = read_lakehouse_table(
    target=TARGET_TARGET, schema=TARGET_SCHEMA, table_name=TARGET_TABLE_NAME, spark_session=spark,
)
target_profile_df = profile_and_register_table(
    target_df, profile_role="target", target=TARGET_TARGET,
    schema=TARGET_SCHEMA, table_name=TARGET_TABLE_NAME,
)
display(target_df)
display(target_profile_df)

### Live — Alternative Warehouse target

In [ ]:
# Run this alternative instead of the Lakehouse target when Warehouse is required.
# TARGET_TARGET = "product"
# TARGET_SCHEMA = "demo"
# TARGET_TABLE_NAME = "email_logs_dummy"
# write_warehouse_table(
#     df=transformed_df, target=TARGET_TARGET, schema=TARGET_SCHEMA,
#     table_name=TARGET_TABLE_NAME, mode="append",
# )
# target_df = read_warehouse_table(
#     target=TARGET_TARGET, schema=TARGET_SCHEMA, table_name=TARGET_TABLE_NAME, spark_session=spark,
# )
# target_profile_df = profile_and_register_table(
#     target_df, profile_role="target", target=TARGET_TARGET,
#     schema=TARGET_SCHEMA, table_name=TARGET_TABLE_NAME,
# )

### Preview — Target Guardrails and governed load preparation
This block belongs immediately before the physical target write. It validates the transformed target, reuses the processing definition resolved during Extract, prepares audit/lifecycle fields and writer settings, writes explicitly, then reads back and profiles the complete persisted target.

In [ ]:
"""
PREVIEW — Enable together with the governed Extract block above.

if read_prep["read_strategy"] != "skip":
    check_schema(
        TARGET_TABLE_NAME, target=TARGET_TARGET, schema=TARGET_SCHEMA, dataframe=transformed_df,
    )
    target_dq_result = check_dq(
        transformed_df, TARGET_TABLE_NAME, target=TARGET_TARGET, schema=TARGET_SCHEMA,
    )
    display(target_dq_result["summary"])
    if not target_dq_result["can_continue"]:
        raise RuntimeError("A DQ Guardrail blocked the target write.")

    write_prep = write_pipeline_prep(transformed_df, read_prep, target=TARGET_TARGET)
    write_lakehouse_table(
        df=write_prep["df"],
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE_NAME,
        mode=write_prep["mode"],
        options=write_prep["options"],
        load_strategy=write_prep["load_strategy"],
        load_strategy_parameters=write_prep["load_strategy_parameters"],
        processing_scope=write_prep["scope"],
    )

    target_df = read_lakehouse_table(
        target=TARGET_TARGET, schema=TARGET_SCHEMA, table_name=TARGET_TABLE_NAME, spark_session=spark,
    )
    registration = dict(
        profile_role="target", target=TARGET_TARGET,
        schema=TARGET_SCHEMA, table_name=TARGET_TABLE_NAME,
    )
    processing = read_prep["processing"]
    if ENV == "dev" and processing["source"] == "current_authoring":
        registration.update(
            load_strategy=TARGET_LOAD_STRATEGY,
            load_strategy_parameters=TARGET_LOAD_PARAMETERS,
        )
    target_profile_df = profile_and_register_table(target_df, **registration)
"""

## Preview — Review this pipeline's governed evidence
The technical self-review is restricted to the active environment, current Fabric workspace, and current notebook lineage.

In [ ]:
"""
pipeline_catalogue_view = widget_view_catalogue(
    mode="pipeline", target="metadata", spark_session=spark,
)
views = pipeline_catalogue_view["get_views"]()
display(views["catalogue"])
display(views["profile"])
display(views["frequency"])
display(views["guardrail_results"])
display(views["guardrail_row_results"])
"""